In [1]:
# ==================== ViT多标签分类训练（极端过采样版）====================

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import gc
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
from transformers import ViTForImageClassification
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 导入你的模块
import sys
sys.path.append('..')

try:
    from src.data.dataset import FundusDataset
    from src.data.focal_loss import FocalLoss
    from src.data.config import config, device
    print("✅ 成功导入自定义模块")
    print(f"使用设备: {device}")
except Exception as e:
    print(f"❌ 导入模块失败: {e}")
    raise

# ========== 数据路径 ==========
train_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Training Images'
test_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Testing Images'
val_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\validation images'
excel_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\raw\ODIR-5K\data.xlsx'

os.makedirs(config['save_dir'], exist_ok=True)
os.makedirs(config['log_dir'], exist_ok=True)

# ========== 分析数据分布 ==========
print("\n📊 分析数据分布...")

train_dataset = FundusDataset(train_dir, excel_dir, is_training=True)
val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

# 收集完整标签统计
def collect_full_labels(dataset, name):
    labels_list = []
    for i in tqdm(range(len(dataset)), desc=f"收集{name}集所有标签"):
        _, labels, _ = dataset[i]
        labels_list.append(labels.numpy())
        if i % 500 == 0:
            gc.collect()
    labels_array = np.vstack(labels_list)
    
    pos_counts = labels_array.sum(axis=0)
    total = len(labels_array)
    
    print(f"\n{name}集完整分布 (共{total}个样本):")
    for i, count in enumerate(pos_counts):
        pct = count / total * 100
        print(f"  类别 {i}: {config['class_names'][i][:10]}: {int(count)} ({pct:.1f}%)")
    
    return labels_array

train_labels = collect_full_labels(train_dataset, "训练")
val_labels = collect_full_labels(val_dataset, "验证")

# ========== 极端过采样策略 ==========
print("\n⚖️ 创建极端过采样策略...")

# 定义各类别的采样权重（少数类权重更高）
class_sample_weights = {
    0: 1.0,   # 正常 (1596样本)
    1: 1.0,   # 糖尿病视网膜病变 (1584样本)
    2: 8.0,   # 青光眼 (304样本)
    3: 8.0,   # 白内障 (296样本)
    4: 15.0,  # 黄斑变性 (232样本)
    5: 20.0,  # 高血压视网膜病变 (146样本)
    6: 8.0,   # 近视 (246样本)
    7: 1.0    # 其他 (1374样本)
}

sample_weights = np.ones(len(train_dataset))

for i in tqdm(range(len(train_dataset)), desc="计算采样权重"):
    _, labels, _ = train_dataset[i]
    
    weight = 1.0
    for class_idx in range(8):
        if labels[class_idx] == 1:
            weight *= class_sample_weights[class_idx]
    
    # 限制权重范围
    sample_weights[i] = np.clip(weight, 0.5, 25.0)

print(f"权重范围: {sample_weights.min():.2f} - {sample_weights.max():.2f}")
print(f"权重均值: {sample_weights.mean():.2f}")

# 过采样倍数提高到5倍
oversample_factor = 5
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights) * oversample_factor,
    replacement=True
)

# ========== DataLoader ==========
batch_size = 16
num_workers = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=num_workers,
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False
)

print(f"训练批次: {len(train_loader)}")
print(f"验证批次: {len(val_loader)}")

# ========== 加载模型 ==========
print("\n🤖 加载预训练模型...")

model = ViTForImageClassification.from_pretrained(
    config['pretrained_path'],
    num_labels=config['num_classes'],
    ignore_mismatched_sizes=True
)

# 更强的Dropout
model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.config.hidden_size, config['num_classes'])
)

model = model.to(device)
print(f"模型参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# ========== 损失函数 ==========
print("\n⚙️ 配置损失函数和优化器...")

# 为少数类设置更高权重
class_loss_weights = torch.tensor([1.0, 1.0, 6.0, 6.0, 12.0, 15.0, 6.0, 1.0], dtype=torch.float32)
class_loss_weights = class_loss_weights.to(device)

class CombinedLoss(nn.Module):
    def __init__(self, class_weights, alpha=0.5, gamma=2.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=class_weights)
        self.focal = FocalLoss(alpha=class_weights, gamma=gamma)
        self.alpha = alpha
    
    def forward(self, inputs, targets):
        return self.alpha * self.bce(inputs, targets) + (1 - self.alpha) * self.focal(inputs, targets)

criterion = CombinedLoss(class_weights=class_loss_weights, alpha=0.5, gamma=2.0)
print("✅ 使用组合损失: 0.5*BCE + 0.5*Focal, gamma=2.0")
print(f"类别权重: {class_loss_weights.cpu().numpy()}")

# 优化器 - 更低的学习率
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-6,
    weight_decay=0.1
)

# 学习率调度器
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3,
    verbose=True,
    min_lr=1e-7
)

# ========== TensorBoard ==========
writer = SummaryWriter(config['log_dir'])
print(f"TensorBoard日志: {config['log_dir']}")

# ========== 训练参数 ==========
best_val_f1 = 0
best_model_state = None
best_thresholds = None
patience_counter = 0
early_stop_patience = 6
epochs = 25
global_step = 0

print("\n🚀 开始训练...")
print(f"总轮数: {epochs}")
print(f"早停patience: {early_stop_patience}")
print("="*60)

# ========== 训练循环 ==========
for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{epochs}")
    print('='*50)
    
    # 训练阶段
    model.train()
    train_loss = 0
    train_steps = 0
    
    train_pbar = tqdm(train_loader, desc='Training')
    for images, labels, _ in train_pbar:
        try:
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss = criterion(outputs.logits, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_steps += 1
            
            writer.add_scalar('Batch/Loss', loss.item(), global_step)
            global_step += 1
            
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
        except Exception as e:
            print(f"训练批次错误: {e}")
            continue
    
    if train_steps == 0:
        print("⚠️ 没有成功训练的批次，请检查数据加载器")
        break
    
    avg_train_loss = train_loss / train_steps
    
    # 验证阶段
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc='Validating'):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    
    # 阈值搜索
    best_thresholds = []
    per_class_best_f1 = []
    
    for i in range(8):
        best_f1 = 0
        best_th = 0.5
        for th in np.arange(0.1, 0.9, 0.05):
            preds = (all_probs[:, i] > th).astype(int)
            f1 = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_th = th
        best_thresholds.append(best_th)
        per_class_best_f1.append(best_f1)
    
    # 使用最佳阈值
    val_preds = np.zeros_like(all_probs)
    for i in range(8):
        val_preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)
    
    # 计算指标
    val_f1_macro = f1_score(all_labels, val_preds, average='macro', zero_division=0)
    val_f1_micro = f1_score(all_labels, val_preds, average='micro', zero_division=0)
    per_class_actual_f1 = f1_score(all_labels, val_preds, average=None, zero_division=0)
    
    # 少数类F1（重点关注类别4和5）
    class4_f1 = per_class_actual_f1[4]
    class5_f1 = per_class_actual_f1[5]
    minority_f1 = np.mean([per_class_actual_f1[i] for i in [2, 3, 4, 5, 6]])
    
    print(f"\n📊 验证结果:")
    print(f"  Loss: {avg_train_loss:.4f}")
    print(f"  Macro F1: {val_f1_macro:.4f}")
    print(f"  Micro F1: {val_f1_micro:.4f}")
    print(f"  类别4(黄斑变性) F1: {class4_f1:.4f}")
    print(f"  类别5(高血压) F1: {class5_f1:.4f}")
    print(f"  少数类平均F1: {minority_f1:.4f}")
    print(f"  各类别F1: {[f'{f:.3f}' for f in per_class_actual_f1]}")
    
    # 记录到TensorBoard
    writer.add_scalar('Train/Loss', avg_train_loss, epoch)
    writer.add_scalar('Val/Macro_F1', val_f1_macro, epoch)
    writer.add_scalar('Val/Class4_F1', class4_f1, epoch)
    writer.add_scalar('Val/Class5_F1', class5_f1, epoch)
    writer.add_scalar('Val/Minority_F1', minority_f1, epoch)
    for i, f1 in enumerate(per_class_actual_f1):
        writer.add_scalar(f'Class_F1/{config["class_names"][i]}', f1, epoch)
    writer.add_scalar('Val/LR', optimizer.param_groups[0]['lr'], epoch)
    
    # 学习率调度
    scheduler.step(val_f1_macro)
    
    # 保存最佳模型
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_thresholds = best_thresholds.copy()
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': best_model_state,
            'val_f1': val_f1_macro,
            'per_class_f1': per_class_actual_f1,
            'best_thresholds': best_thresholds,
            'config': config
        }
        
        torch.save(checkpoint, os.path.join(config['save_dir'], 'best_model.pth'))
        print(f"  ✅ 保存最佳模型! F1={val_f1_macro:.4f}")
    else:
        patience_counter += 1
        print(f"  ⏳ 早停计数: {patience_counter}/{early_stop_patience}")
    
    # 清理内存
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    if patience_counter >= early_stop_patience:
        print(f"\n⏹️ 早停: {early_stop_patience}轮未提升")
        break

# ========== 最终测试 ==========
print("\n" + "="*60)
print("🧪 最终测试")
print("="*60)

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.eval()
    
    test_probs = []
    test_labels = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            test_probs.append(probs)
            test_labels.append(labels.numpy())
    
    test_probs = np.vstack(test_probs)
    test_labels = np.vstack(test_labels)
    
    test_preds = np.zeros_like(test_probs)
    for i in range(8):
        test_preds[:, i] = (test_probs[:, i] > best_thresholds[i]).astype(int)
    
    test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    test_per_class_f1 = f1_score(test_labels, test_preds, average=None, zero_division=0)
    
    print(f"\n📊 最终测试结果:")
    print(f"  Macro F1: {test_f1_macro:.4f}")
    print(f"  类别4(黄斑变性) F1: {test_per_class_f1[4]:.4f}")
    print(f"  类别5(高血压) F1: {test_per_class_f1[5]:.4f}")
    print(f"  各类别F1: {[f'{f:.3f}' for f in test_per_class_f1]}")
    
    # 保存结果
    results_df = pd.DataFrame({
        'Class': config['class_names'],
        'F1_Score': test_per_class_f1,
        'Threshold': best_thresholds
    })
    results_df.to_csv('test_results.csv', index=False, encoding='utf-8-sig')
    
    print(f"\n✅ 训练完成！最佳验证F1: {best_val_f1:.4f}")
    print(f"最终测试F1: {test_f1_macro:.4f}")

writer.close()
print(f"\nTensorBoard: tensorboard --logdir {config['log_dir']}")

✅ 成功导入自定义模块
使用设备: cuda

📊 分析数据分布...
训练集大小: 4906
验证集大小: 1046
测试集大小: 1048


收集训练集所有标签: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4906/4906 [00:42<00:00, 114.85it/s]



训练集完整分布 (共4906个样本):
  类别 0: 正常: 1596 (32.5%)
  类别 1: 糖尿病视网膜病变: 1584 (32.3%)
  类别 2: 青光眼: 304 (6.2%)
  类别 3: 白内障: 296 (6.0%)
  类别 4: 黄斑变性: 232 (4.7%)
  类别 5: 高血压视网膜病变: 146 (3.0%)
  类别 6: 近视: 246 (5.0%)
  类别 7: 其他: 1374 (28.0%)


收集验证集所有标签: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1046/1046 [00:05<00:00, 179.77it/s]



验证集完整分布 (共1046个样本):
  类别 0: 正常: 342 (32.7%)
  类别 1: 糖尿病视网膜病变: 336 (32.1%)
  类别 2: 青光眼: 62 (5.9%)
  类别 3: 白内障: 62 (5.9%)
  类别 4: 黄斑变性: 50 (4.8%)
  类别 5: 高血压视网膜病变: 30 (2.9%)
  类别 6: 近视: 50 (4.8%)
  类别 7: 其他: 288 (27.5%)

⚖️ 创建极端过采样策略...


...


📊 验证结果:
  Loss: 0.2463
  Macro F1: 0.4564
  Micro F1: 0.5123
  类别4(黄斑变性) F1: 0.1860
  类别5(高血压) F1: 0.0930
  少数类平均F1: 0.4135
  各类别F1: ['0.577', '0.568', '0.259', '0.734', '0.186', '0.093', '0.795', '0.439']
  ⏳ 早停计数: 5/6

Epoch 8/25


Validating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66/66 [00:16<00:00,  4.01it/s]



📊 验证结果:
  Loss: 0.2190
  Macro F1: 0.4393
  Micro F1: 0.5139
  类别4(黄斑变性) F1: 0.0968
  类别5(高血压) F1: 0.0571
  少数类平均F1: 0.3828
  各类别F1: ['0.582', '0.582', '0.255', '0.710', '0.097', '0.057', '0.795', '0.437']
  ⏳ 早停计数: 6/6

⏹️ 早停: 6轮未提升

🧪 最终测试


Testing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66/66 [00:16<00:00,  3.91it/s]


📊 最终测试结果:
  Macro F1: 0.4025
  类别4(黄斑变性) F1: 0.2479
  类别5(高血压) F1: 0.0000
  各类别F1: ['0.414', '0.514', '0.251', '0.602', '0.248', '0.000', '0.752', '0.439']

✅ 训练完成！最佳验证F1: 0.4997
最终测试F1: 0.4025

TensorBoard: tensorboard --logdir ./logs


In [1]:
# ==================== ViT多标签分类训练（调用已有模块）====================
# 运行前请确保已安装：pip install transformers tensorboard scikit-learn

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import time
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 导入你的模块
import sys
sys.path.append('..')  # 添加上级目录到路径，根据你的项目结构调整

try:
    from src.data.dataset import FundusDataset
    from src.data.focal_loss import FocalLoss
    from src.data.config import config, device
    print("✅ 成功导入自定义模块")
except Exception as e:
    print(f"❌ 导入模块失败: {e}")
    print("请确保路径正确，或手动调整sys.path")
    raise

print("="*60)
print("ViT多标签分类训练开始")
print("="*60)

# ========== 1. 设备配置 ==========
print(f"使用设备: {device}")

# ========== 2. 数据路径 ==========
train_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Training Images'
test_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Testing Images'
val_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\validation images'
excel_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\raw\ODIR-5K\data.xlsx'

# 创建保存目录
os.makedirs(config['save_dir'], exist_ok=True)
os.makedirs(config['log_dir'], exist_ok=True)

# ========== 3. 分析数据分布 ==========
print("\n📊 分析数据分布...")

# 加载数据集
train_dataset = FundusDataset(train_dir, excel_dir, is_training=True)
val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

# 收集标签统计
def collect_label_stats(dataset, name):
    labels_list = []
    for i in tqdm(range(min(len(dataset), 3000)), desc=f"分析{name}集"):
        _, labels, _ = dataset[i]
        labels_list.append(labels.numpy())
    labels_array = np.vstack(labels_list)
    
    pos_counts = labels_array.sum(axis=0)
    print(f"\n{name}集正样本分布:")
    for i, count in enumerate(pos_counts):
        pct = count / len(labels_array) * 100
        print(f"  类别 {i}: {config['class_names'][i][:10]}: {int(count)} ({pct:.1f}%)")
    
    return pos_counts

train_pos_counts = collect_label_stats(train_dataset, "训练")
val_pos_counts = collect_label_stats(val_dataset, "验证")

# 确定少数类（正样本数少的类别）
minority_threshold = np.percentile(train_pos_counts, 33)  # 后33%为少数类
minority_classes = [i for i, count in enumerate(train_pos_counts) if count <= minority_threshold]
print(f"\n少数类索引: {minority_classes}")
print(f"少数类名称: {[config['class_names'][i] for i in minority_classes]}")

# ========== 4. 创建采样器 ==========
print("\n⚖️ 创建加权采样器...")

# 计算每个样本的权重
sample_weights = np.ones(len(train_dataset))
pos_counts = train_pos_counts

for i in tqdm(range(len(train_dataset)), desc="计算采样权重"):
    _, labels, _ = train_dataset[i]
    
    weight = 1.0
    # 基于类别频率的权重
    for class_idx in range(8):
        if labels[class_idx] == 1:
            class_weight = len(train_dataset) / (pos_counts[class_idx] + 1)
            weight *= class_weight ** 0.3
    
    # 少数类额外权重
    for class_idx in minority_classes:
        if labels[class_idx] == 1:
            weight *= 3.0
    
    sample_weights[i] = np.clip(weight, 0.1, 30)

print(f"权重范围: {sample_weights.min():.2f} - {sample_weights.max():.2f}")
print(f"权重均值: {sample_weights.mean():.2f}")

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights) * 2,
    replacement=True
)

# ========== 5. 创建DataLoader ==========
print("\n🔄 创建DataLoader...")

batch_size = config.get('batch_size', 8)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=0,
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print(f"训练批次: {len(train_loader)}")
print(f"验证批次: {len(val_loader)}")

# ========== 6. 加载模型 ==========
print("\n🤖 加载预训练模型...")

processor = ViTImageProcessor.from_pretrained(config['pretrained_path'])
model = ViTForImageClassification.from_pretrained(
    config['pretrained_path'],
    num_labels=config['num_classes'],
    ignore_mismatched_sizes=True
)
model = model.to(device)
print(f"模型参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# ========== 7. 损失函数和优化器 ==========
print("\n⚙️ 配置损失函数和优化器...")

# 计算类别权重
neg_counts = len(train_dataset) - train_pos_counts
pos_weight = torch.tensor(neg_counts / (train_pos_counts + 1e-5), dtype=torch.float32).to(device)
pos_weight = torch.clamp(pos_weight, 0.1, 10.0)

print("类别权重:")
for i, w in enumerate(pos_weight):
    print(f"  类别 {i}: {config['class_names'][i][:10]}: {w:.3f}")

# 组合损失
criterion_bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion_focal = FocalLoss(alpha=pos_weight, gamma=2.0)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.get('learning_rate', 2e-5),
    weight_decay=config.get('weight_decay', 0.01)
)

scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=5,
    T_mult=2,
    eta_min=1e-6
)

# ========== 8. TensorBoard ==========
writer = SummaryWriter(config['log_dir'])
print(f"TensorBoard日志: {config['log_dir']}")

# ========== 9. 训练循环 ==========
print("\n🚀 开始训练...")
print("="*60)

best_val_f1 = 0
patience_counter = 0
early_stop_patience = 10
epochs = config.get('epochs', 30)
global_step = 0

for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{epochs}")
    print('='*50)
    
    # 训练阶段
    model.train()
    train_loss_bce = 0
    train_loss_focal = 0
    train_steps = 0
    
    train_pbar = tqdm(train_loader, desc='Training')
    for images, labels, _ in train_pbar:
        try:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss_bce = criterion_bce(outputs.logits, labels)
            loss_focal = criterion_focal(outputs.logits, labels)
            loss = loss_bce + 0.5 * loss_focal
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
            optimizer.step()
            
            train_loss_bce += loss_bce.item()
            train_loss_focal += loss_focal.item()
            train_steps += 1
            
            # 记录batch
            writer.add_scalar('Batch/Loss_BCE', loss_bce.item(), global_step)
            writer.add_scalar('Batch/Loss_Focal', loss_focal.item(), global_step)
            writer.add_scalar('Batch/Loss_Total', loss.item(), global_step)
            global_step += 1
            
            train_pbar.set_postfix({
                'bce': f'{loss_bce.item():.4f}',
                'focal': f'{loss_focal.item():.4f}'
            })
            
        except Exception as e:
            print(f"训练批次错误: {e}")
            continue
    
    avg_loss_bce = train_loss_bce / train_steps
    avg_loss_focal = train_loss_focal / train_steps
    
    # 验证阶段
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc='Validating'):
            try:
                images = images.to(device)
                outputs = model(images)
                probs = torch.sigmoid(outputs.logits).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(labels.numpy())
            except Exception as e:
                print(f"验证批次错误: {e}")
                continue
    
    if len(all_probs) == 0:
        print("警告: 验证集无有效数据")
        continue
    
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    
    # 为每个类别寻找最佳阈值
    best_thresholds = []
    per_class_best_f1 = []
    
    for i in range(8):
        best_f1 = 0
        best_th = 0.5
        for th in np.arange(0.2, 0.9, 0.05):
            preds = (all_probs[:, i] > th).astype(int)
            f1 = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_th = th
        best_thresholds.append(best_th)
        per_class_best_f1.append(best_f1)
    
    # 使用最佳阈值
    val_preds = np.zeros_like(all_probs)
    for i in range(8):
        val_preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)
    
    # 计算指标
    val_f1_macro = f1_score(all_labels, val_preds, average='macro', zero_division=0)
    val_f1_micro = f1_score(all_labels, val_preds, average='micro', zero_division=0)
    per_class_f1 = f1_score(all_labels, val_preds, average=None, zero_division=0)
    
    # 少数类F1
    if len(minority_classes) > 0:
        minority_f1 = np.mean([per_class_f1[i] for i in minority_classes])
    else:
        minority_f1 = 0
    
    # 打印结果
    print(f"\n📊 验证结果:")
    print(f"  Loss BCE: {avg_loss_bce:.4f}, Focal: {avg_loss_focal:.4f}")
    print(f"  Macro F1: {val_f1_macro:.4f}")
    print(f"  Micro F1: {val_f1_micro:.4f}")
    print(f"  少数类F1: {minority_f1:.4f}")
    print(f"  最佳阈值: {[f'{th:.2f}' for th in best_thresholds]}")
    
    # 记录到TensorBoard
    writer.add_scalar('Train/Loss_BCE', avg_loss_bce, epoch)
    writer.add_scalar('Train/Loss_Focal', avg_loss_focal, epoch)
    writer.add_scalar('Val/Macro_F1', val_f1_macro, epoch)
    writer.add_scalar('Val/Micro_F1', val_f1_micro, epoch)
    writer.add_scalar('Val/Minority_F1', minority_f1, epoch)
    writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)
    
    # 记录每个类别的F1
    for i, f1 in enumerate(per_class_f1):
        writer.add_scalar(f'Class_F1/{config["class_names"][i]}', f1, epoch)
    
    scheduler.step()
    
    # 保存最佳模型
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1_macro,
            'per_class_f1': per_class_f1,
            'best_thresholds': best_thresholds,
            'minority_classes': minority_classes,
            'config': config
        }
        
        torch.save(checkpoint, os.path.join(config['save_dir'], 'best_model.pth'))
        print(f"  ✅ 保存最佳模型! F1={val_f1_macro:.4f}")
    else:
        patience_counter += 1
    
    # 早停
    if patience_counter >= early_stop_patience:
        print(f"\n⏹️ 早停: {early_stop_patience}轮未提升")
        break

# ========== 10. 最终测试 ==========
print("\n" + "="*60)
print("🧪 最终测试")
print("="*60)

# 加载最佳模型
checkpoint = torch.load(os.path.join(config['save_dir'], 'best_model.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
best_thresholds = checkpoint['best_thresholds']

model.eval()
test_probs = []
test_labels = []

with torch.no_grad():
    for images, labels, _ in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        test_probs.append(probs)
        test_labels.append(labels.numpy())

test_probs = np.vstack(test_probs)
test_labels = np.vstack(test_labels)

# 使用最佳阈值
test_preds = np.zeros_like(test_probs)
for i in range(8):
    test_preds[:, i] = (test_probs[:, i] > best_thresholds[i]).astype(int)

# 计算最终指标
test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
test_f1_micro = f1_score(test_labels, test_preds, average='micro', zero_division=0)
per_class_test_f1 = f1_score(test_labels, test_preds, average=None, zero_division=0)

print(f"\n📊 最终测试结果:")
print(f"  Macro F1: {test_f1_macro:.4f}")
print(f"  Micro F1: {test_f1_micro:.4f}")

print("\n📈 各类别详细指标:")
for i in range(8):
    precision = precision_score(test_labels[:, i], test_preds[:, i], zero_division=0)
    recall = recall_score(test_labels[:, i], test_preds[:, i], zero_division=0)
    print(f"  {config['class_names'][i][:10]:10}: F1={per_class_test_f1[i]:.3f}, "
          f"P={precision:.3f}, R={recall:.3f}, Th={best_thresholds[i]:.2f}")

# 保存结果
results_df = pd.DataFrame({
    'Class': config['class_names'],
    'F1_Score': per_class_test_f1,
    'Precision': [precision_score(test_labels[:, i], test_preds[:, i], zero_division=0) for i in range(8)],
    'Recall': [recall_score(test_labels[:, i], test_preds[:, i], zero_division=0) for i in range(8)],
    'Threshold': best_thresholds
})
results_df.to_csv('test_results.csv', index=False, encoding='utf-8-sig')

writer.close()
print(f"\n✅ 训练完成！最佳验证F1: {best_val_f1:.4f}")
print(f"最终测试F1: {test_f1_macro:.4f}")
print(f"结果已保存到 test_results.csv")
print(f"TensorBoard: tensorboard --logdir {config['log_dir']}")

✅ 成功导入自定义模块
ViT多标签分类训练开始
使用设备: cuda

📊 分析数据分布...
训练集大小: 4906
验证集大小: 1046
测试集大小: 1048


分析训练集: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [00:23<00:00, 125.34it/s]



训练集正样本分布:
  类别 0: 正常: 1574 (52.5%)
  类别 1: 糖尿病视网膜病变: 260 (8.7%)
  类别 2: 青光眼: 294 (9.8%)
  类别 3: 白内障: 256 (8.5%)
  类别 4: 黄斑变性: 148 (4.9%)
  类别 5: 高血压视网膜病变: 66 (2.2%)
  类别 6: 近视: 216 (7.2%)
  类别 7: 其他: 634 (21.1%)


分析验证集: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1046/1046 [00:04<00:00, 238.04it/s]



验证集正样本分布:
  类别 0: 正常: 342 (32.7%)
  类别 1: 糖尿病视网膜病变: 336 (32.1%)
  类别 2: 青光眼: 62 (5.9%)
  类别 3: 白内障: 62 (5.9%)
  类别 4: 黄斑变性: 50 (4.8%)
  类别 5: 高血压视网膜病变: 30 (2.9%)
  类别 6: 近视: 50 (4.8%)
  类别 7: 其他: 288 (27.5%)

少数类索引: [4, 5, 6]
少数类名称: ['黄斑变性', '高血压视网膜病变', '近视']

⚖️ 创建加权采样器...


...


📊 验证结果:
  Loss BCE: 0.2355, Focal: 0.1902
  Macro F1: 0.4345
  Micro F1: 0.5274
  少数类F1: 0.3224
  最佳阈值: ['0.25', '0.65', '0.20', '0.20', '0.20', '0.20', '0.45', '0.35']

Epoch 17/25


Validating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66/66 [00:17<00:00,  3.82it/s]



📊 验证结果:
  Loss BCE: 0.2129, Focal: 0.1750
  Macro F1: 0.4060
  Micro F1: 0.5122
  少数类F1: 0.2820
  最佳阈值: ['0.25', '0.50', '0.20', '0.20', '0.20', '0.50', '0.35', '0.40']

⏹️ 早停: 10轮未提升

🧪 最终测试


Testing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66/66 [00:17<00:00,  3.67it/s]



📊 最终测试结果:
  Macro F1: 0.4683
  Micro F1: 0.5101

📈 各类别详细指标:
  正常        : F1=0.539, P=0.435, R=0.708, Th=0.40
  糖尿病视网膜病变  : F1=0.567, P=0.527, R=0.613, Th=0.70
  青光眼       : F1=0.368, P=0.420, R=0.328, Th=0.20
  白内障       : F1=0.661, P=0.837, R=0.545, Th=0.25
  黄斑变性      : F1=0.280, P=0.277, R=0.283, Th=0.20
  高血压视网膜病变  : F1=0.043, P=0.059, R=0.033, Th=0.20
  近视        : F1=0.842, P=0.930, R=0.769, Th=0.25
  其他        : F1=0.448, P=0.336, R=0.672, Th=0.50

✅ 训练完成！最佳验证F1: 0.4889
最终测试F1: 0.4683
结果已保存到 test_results.csv
TensorBoard: tensorboard --logdir ./logs


In [2]:
# ==================== ViT多标签分类训练（修复数据分布问题）====================

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
from transformers import ViTForImageClassification
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 导入你的模块
import sys
sys.path.append('..')

try:
    from src.data.dataset import FundusDataset
    from src.data.focal_loss import FocalLoss
    from src.data.config import config, device
    print("✅ 成功导入自定义模块")
except Exception as e:
    print(f"❌ 导入模块失败: {e}")
    raise

print("="*60)
print("ViT多标签分类训练开始（修复版）")
print("="*60)
print(f"使用设备: {device}")

# ========== 数据路径 ==========
train_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Training Images'
test_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Testing Images'
val_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\validation images'
excel_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\raw\ODIR-5K\data.xlsx'

os.makedirs(config['save_dir'], exist_ok=True)
os.makedirs(config['log_dir'], exist_ok=True)

# ========== 分析数据分布 ==========
print("\n📊 分析数据分布...")

train_dataset = FundusDataset(train_dir, excel_dir, is_training=True)
val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

# 收集完整标签统计
def collect_full_labels(dataset, name):
    labels_list = []
    for i in tqdm(range(len(dataset)), desc=f"收集{name}集所有标签"):
        _, labels, _ = dataset[i]
        labels_list.append(labels.numpy())
    labels_array = np.vstack(labels_list)
    
    pos_counts = labels_array.sum(axis=0)
    total = len(labels_array)
    
    print(f"\n{name}集完整分布 (共{total}个样本):")
    for i, count in enumerate(pos_counts):
        pct = count / total * 100
        print(f"  类别 {i}: {config['class_names'][i][:10]}: {int(count)} ({pct:.1f}%)")
    
    return labels_array

train_labels = collect_full_labels(train_dataset, "训练")
val_labels = collect_full_labels(val_dataset, "验证")

# 计算每个类别的权重（基于验证集分布调整）
def compute_balanced_weights(train_labels, val_labels):
    """根据验证集分布调整训练权重"""
    train_pos = train_labels.sum(axis=0)
    val_pos = val_labels.sum(axis=0)
    train_total = len(train_labels)
    val_total = len(val_labels)
    
    # 目标分布：让训练集更接近验证集
    target_dist = val_pos / val_total
    current_dist = train_pos / train_total
    
    # 计算调整因子
    adjust_factor = target_dist / (current_dist + 1e-8)
    adjust_factor = np.clip(adjust_factor, 0.2, 5.0)  # 限制范围
    
    print("\n📈 分布调整因子:")
    for i, factor in enumerate(adjust_factor):
        print(f"  类别 {i}: train={current_dist[i]:.3f}, val={target_dist[i]:.3f}, factor={factor:.2f}")
    
    return adjust_factor

adjust_factor = compute_balanced_weights(train_labels, val_labels)

# ========== 改进的采样策略 ==========
print("\n⚖️ 创建自适应采样器...")

# 基于验证集分布计算样本权重
sample_weights = np.ones(len(train_dataset))
train_pos_counts = train_labels.sum(axis=0)

for i in tqdm(range(len(train_dataset)), desc="计算采样权重"):
    _, labels, _ = train_dataset[i]
    
    weight = 1.0
    for class_idx in range(8):
        if labels[class_idx] == 1:
            # 使用调整后的权重
            class_weight = adjust_factor[class_idx] * (len(train_dataset) / (train_pos_counts[class_idx] + 1))
            weight *= class_weight ** 0.5
    
    # 对极少数类额外加权
    minority_classes = [4, 5, 6]  # 黄斑变性、高血压、近视
    for class_idx in minority_classes:
        if labels[class_idx] == 1:
            weight *= 2.0
    
    sample_weights[i] = np.clip(weight, 0.5, 20.0)

print(f"权重范围: {sample_weights.min():.2f} - {sample_weights.max():.2f}")
print(f"权重均值: {sample_weights.mean():.2f}")

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights) * 3,  # 增加采样倍数
    replacement=True
)

# ========== DataLoader ==========
batch_size = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=4,
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=False
)

print(f"训练批次: {len(train_loader)}")
print(f"验证批次: {len(val_loader)}")

# ========== 加载模型 ==========
print("\n🤖 加载预训练模型...")

model = ViTForImageClassification.from_pretrained(
    config['pretrained_path'],
    num_labels=config['num_classes'],
    ignore_mismatched_sizes=True
)
model = model.to(device)
print(f"模型参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# ========== 损失函数 ==========
print("\n⚙️ 配置损失函数和优化器...")

# 使用验证集分布计算类别权重
val_pos_counts = val_labels.sum(axis=0)
val_neg_counts = len(val_labels) - val_pos_counts
pos_weight = torch.tensor(val_neg_counts / (val_pos_counts + 1e-5), dtype=torch.float32).to(device)
pos_weight = torch.clamp(pos_weight, 0.5, 10.0)

print("类别权重（基于验证集）:")
for i, w in enumerate(pos_weight):
    print(f"  类别 {i}: {config['class_names'][i][:10]}: {w:.3f}")

# 组合损失
criterion_bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion_focal = FocalLoss(alpha=pos_weight, gamma=2.0)

# 优化器（更低的学习率）
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,  # 降低学习率
    weight_decay=0.05  # 增加正则化
)

# 使用ReduceLROnPlateau，当验证F1不提升时降低学习率
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3,
    verbose=True,
    min_lr=1e-7
)

# ========== TensorBoard ==========
writer = SummaryWriter(config['log_dir'])
print(f"TensorBoard日志: {config['log_dir']}")

# ========== 训练循环 ==========
print("\n🚀 开始训练...")
print("="*60)

best_val_f1 = 0
best_model_state = None
best_thresholds = None
patience_counter = 0
early_stop_patience = 15
epochs = 30
global_step = 0

for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{epochs}")
    print('='*50)
    
    # 训练阶段
    model.train()
    train_loss_bce = 0
    train_loss_focal = 0
    train_steps = 0
    
    train_pbar = tqdm(train_loader, desc='Training')
    for images, labels, _ in train_pbar:
        try:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss_bce = criterion_bce(outputs.logits, labels)
            loss_focal = criterion_focal(outputs.logits, labels)
            loss = loss_bce + 0.3 * loss_focal  # 降低focal loss权重
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
            optimizer.step()
            
            train_loss_bce += loss_bce.item()
            train_loss_focal += loss_focal.item()
            train_steps += 1
            
            writer.add_scalar('Batch/Loss_Total', loss.item(), global_step)
            global_step += 1
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}'
            })
            
        except Exception as e:
            print(f"训练批次错误: {e}")
            continue
    
    avg_loss = (train_loss_bce + train_loss_focal) / train_steps
    
    # 验证阶段
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc='Validating'):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    
    # 阈值搜索
    best_thresholds = []
    per_class_f1 = []
    
    for i in range(8):
        best_f1 = 0
        best_th = 0.5
        for th in np.arange(0.2, 0.8, 0.05):
            preds = (all_probs[:, i] > th).astype(int)
            f1 = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_th = th
        best_thresholds.append(best_th)
        per_class_f1.append(best_f1)
    
    # 使用最佳阈值
    val_preds = np.zeros_like(all_probs)
    for i in range(8):
        val_preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)
    
    # 计算指标
    val_f1_macro = f1_score(all_labels, val_preds, average='macro', zero_division=0)
    val_f1_micro = f1_score(all_labels, val_preds, average='micro', zero_division=0)
    per_class_actual_f1 = f1_score(all_labels, val_preds, average=None, zero_division=0)
    
    # 少数类F1
    minority_f1 = np.mean([per_class_actual_f1[i] for i in [4,5,6]])
    
    print(f"\n📊 验证结果:")
    print(f"  Loss: {avg_loss:.4f}")
    print(f"  Macro F1: {val_f1_macro:.4f}")
    print(f"  Micro F1: {val_f1_micro:.4f}")
    print(f"  少数类F1: {minority_f1:.4f}")
    print(f"  各类别F1: {[f'{f:.3f}' for f in per_class_actual_f1]}")
    
    # 记录
    writer.add_scalar('Train/Loss', avg_loss, epoch)
    writer.add_scalar('Val/Macro_F1', val_f1_macro, epoch)
    writer.add_scalar('Val/Minority_F1', minority_f1, epoch)
    for i, f1 in enumerate(per_class_actual_f1):
        writer.add_scalar(f'Class_F1/{config["class_names"][i]}', f1, epoch)
    
    # 学习率调度
    scheduler.step(val_f1_macro)
    
    # 保存最佳模型
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_thresholds = best_thresholds.copy()
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': best_model_state,
            'val_f1': val_f1_macro,
            'per_class_f1': per_class_actual_f1,
            'best_thresholds': best_thresholds,
            'config': config
        }
        
        torch.save(checkpoint, os.path.join(config['save_dir'], 'best_model.pth'))
        print(f"  ✅ 保存最佳模型! F1={val_f1_macro:.4f}")
    else:
        patience_counter += 1
        print(f"  ⏳ 早停计数: {patience_counter}/{early_stop_patience}")
    
    if patience_counter >= early_stop_patience:
        print(f"\n⏹️ 早停: {early_stop_patience}轮未提升")
        break

# ========== 最终测试 ==========
print("\n" + "="*60)
print("🧪 最终测试")
print("="*60)

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.eval()
    
    test_probs = []
    test_labels = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            test_probs.append(probs)
            test_labels.append(labels.numpy())
    
    test_probs = np.vstack(test_probs)
    test_labels = np.vstack(test_labels)
    
    test_preds = np.zeros_like(test_probs)
    for i in range(8):
        test_preds[:, i] = (test_probs[:, i] > best_thresholds[i]).astype(int)
    
    test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    
    print(f"\n📊 最终测试结果:")
    print(f"  Macro F1: {test_f1_macro:.4f}")
    
    # 保存结果
    results_df = pd.DataFrame({
        'Class': config['class_names'],
        'F1_Score': f1_score(test_labels, test_preds, average=None, zero_division=0),
        'Threshold': best_thresholds
    })
    results_df.to_csv('test_results.csv', index=False, encoding='utf-8-sig')
    
    print(f"\n✅ 训练完成！最佳验证F1: {best_val_f1:.4f}")
    print(f"最终测试F1: {test_f1_macro:.4f}")

writer.close()
print(f"\nTensorBoard: tensorboard --logdir {config['log_dir']}")

✅ 成功导入自定义模块
ViT多标签分类训练开始（修复版）
使用设备: cuda

📊 分析数据分布...
训练集大小: 4906
验证集大小: 1046
测试集大小: 1048


收集训练集所有标签: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4906/4906 [00:35<00:00, 136.58it/s]



训练集完整分布 (共4906个样本):
  类别 0: 正常: 1596 (32.5%)
  类别 1: 糖尿病视网膜病变: 1584 (32.3%)
  类别 2: 青光眼: 304 (6.2%)
  类别 3: 白内障: 296 (6.0%)
  类别 4: 黄斑变性: 232 (4.7%)
  类别 5: 高血压视网膜病变: 146 (3.0%)
  类别 6: 近视: 246 (5.0%)
  类别 7: 其他: 1374 (28.0%)


收集验证集所有标签: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1046/1046 [00:04<00:00, 214.84it/s]



验证集完整分布 (共1046个样本):
  类别 0: 正常: 342 (32.7%)
  类别 1: 糖尿病视网膜病变: 336 (32.1%)
  类别 2: 青光眼: 62 (5.9%)
  类别 3: 白内障: 62 (5.9%)
  类别 4: 黄斑变性: 50 (4.8%)
  类别 5: 高血压视网膜病变: 30 (2.9%)
  类别 6: 近视: 50 (4.8%)
  类别 7: 其他: 288 (27.5%)

📈 分布调整因子:
  类别 0: train=0.325, val=0.327, factor=1.01
  类别 1: train=0.323, val=0.321, factor=0.99
  类别 2: train=0.062, val=0.059, factor=0.96
  类别 3: train=0.060, val=0.059, factor=0.98
  类别 4: train=0.047, val=0.048, factor=1.01
  类别 5: train=0.030, val=0.029, factor=0.96
  类别 6: train=0.050, val=0.048, factor=0.95
  类别 7: train=0.280, val=0.275, factor=0.98

⚖️ 创建自适应采样器...


...


📊 验证结果:
  Loss: 0.0323
  Macro F1: 0.3258
  Micro F1: 0.4765
  少数类F1: 0.2222
  各类别F1: ['0.524', '0.553', '0.029', '0.410', '0.000', '0.000', '0.667', '0.423']
  ⏳ 早停计数: 14/15

Epoch 17/30


Validating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 131/131 [00:29<00:00,  4.51it/s]



📊 验证结果:
  Loss: 0.0281
  Macro F1: 0.3231
  Micro F1: 0.4708
  少数类F1: 0.2105
  各类别F1: ['0.529', '0.529', '0.058', '0.410', '0.000', '0.000', '0.632', '0.427']
  ⏳ 早停计数: 15/15

⏹️ 早停: 15轮未提升

🧪 最终测试


Testing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 131/131 [00:29<00:00,  4.51it/s]


📊 最终测试结果:
  Macro F1: 0.4353

✅ 训练完成！最佳验证F1: 0.4744
最终测试F1: 0.4353

TensorBoard: tensorboard --logdir ./logs


In [ ]:
# ==================== 单元格2: 数据增强和Dataset类 ====================

# 数据增强
def transform(is_training=True):
    if is_training:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

# 强数据增强
def strong_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
        transforms.RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0)),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

# Dataset类
class FundusDataset(Dataset):
    def __init__(self, img_dir, excel_path, is_training=True):
        self.img_dir = img_dir
        self.transform = transform(is_training)
        self.strong_transform = strong_transform() if is_training else None
        self.is_training = is_training
        self.diseases = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
        self.disease_names = {
            'N': '正常',
            'D': '糖尿病视网膜病变',
            'G': '青光眼',
            'C': '白内障',
            'A': '年龄相关性黄斑变性',
            'H': '高血压视网膜病变',
            'M': '病理性近视',
            'O': '其他疾病'
        }
        
        # 获取所有图片
        self.image_files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
        self.df = pd.read_excel(excel_path)
        self.label_map = self._create_label_map()
        
        # 预计算少数类标记
        self.minority_classes = []  # 将在分析后设置
        self.is_minority_cache = {}

    def set_minority_classes(self, minority_classes):
        """设置少数类并预计算"""
        self.minority_classes = minority_classes
        self._precompute_minority_flags()

    def _precompute_minority_flags(self):
        """预计算每个样本是否为少数类"""
        for img_name in self.image_files:
            if img_name in self.label_map:
                labels = self.label_map[img_name]
                is_minority = False
                for class_idx in self.minority_classes:
                    if class_idx < len(labels) and labels[class_idx] == 1:
                        is_minority = True
                        break
                self.is_minority_cache[img_name] = is_minority
            else:
                self.is_minority_cache[img_name] = False

    def __len__(self):
        return len(self.image_files)

    def _create_label_map(self):
        label_map = {}
        for index, row in self.df.iterrows():
            if pd.notna(row['Left-Fundus']):
                labels = []
                for disease in self.diseases:
                    value = row[disease]
                    labels.append(value)
                label_map[row['Left-Fundus']] = np.array(labels, dtype=np.float32)
            
            if pd.notna(row['Right-Fundus']):
                labels = []
                for disease in self.diseases:
                    value = row[disease]
                    labels.append(value)
                label_map[row['Right-Fundus']] = np.array(labels, dtype=np.float32)
        return label_map

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            print(f"警告: 无法读取图片 {img_path}，使用空白图片")
            image = Image.new('RGB', (224, 224), color='black')
        
        # 获取标签
        if img_name in self.label_map:
            labels = self.label_map[img_name]
        else:
            labels = np.zeros(len(self.diseases), dtype=np.float32)
        
        # 数据增强
        if self.is_training:
            is_minority = self.is_minority_cache.get(img_name, False)
            if is_minority and self.strong_transform is not None:
                image = self.strong_transform(image)
            else:
                image = self.transform(image)
        else:
            image = self.transform(image)
        
        labels = torch.FloatTensor(labels)
        return image, labels, img_name

print("✅ Dataset类定义完成")

In [ ]:
# ==================== 单元格3: 损失函数定义 ====================

class FocalLoss(nn.Module):
    """Focal Loss for multi-label classification"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = (1 - pt) ** self.gamma * bce_loss
        
        if self.alpha is not None:
            if self.alpha.dim() == 1:
                alpha = self.alpha.view(1, -1).expand_as(focal_loss)
            else:
                alpha = self.alpha
            focal_loss = focal_loss * alpha
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class AdaptiveFocalLoss(nn.Module):
    """自适应Focal Loss，根据类别表现调整gamma"""
    def __init__(self, class_names, gamma=2.0, alpha=None):
        super().__init__()
        self.class_names = class_names
        self.base_gamma = gamma
        self.gamma = torch.ones(8) * gamma
        self.alpha = alpha
        self.performance_history = []
        
    def update_gamma(self, class_f1_scores):
        """根据各类别F1分数调整gamma"""
        self.performance_history.append(class_f1_scores)
        avg_f1 = np.mean(class_f1_scores)
        
        for i, f1 in enumerate(class_f1_scores):
            if f1 < avg_f1 * 0.8:
                self.gamma[i] = min(5.0, self.base_gamma * 1.5)
            elif f1 > avg_f1 * 1.2:
                self.gamma[i] = max(1.0, self.base_gamma * 0.8)
            else:
                self.gamma[i] = self.base_gamma
        
        print("\n更新gamma值:")
        for i, g in enumerate(self.gamma):
            print(f"  {self.class_names[i]}: {g:.2f} (F1={class_f1_scores[i]:.3f})")
    
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        
        gamma_tensor = self.gamma.view(1, -1).to(inputs.device)
        focal_loss = (1 - pt) ** gamma_tensor * bce_loss
        
        if self.alpha is not None:
            alpha_t = self.alpha.view(1, -1).to(inputs.device)
            focal_loss = focal_loss * alpha_t
        
        return focal_loss.mean()

print("✅ 损失函数定义完成")

In [ ]:
# ==================== 单元格4: 阈值优化器 ====================

class ThresholdOptimizer:
    """为每个类别优化阈值"""
    def __init__(self, class_names):
        self.class_names = class_names
        self.best_thresholds = [0.5] * 8
        self.threshold_history = []
        
    def find_best_thresholds(self, val_probs, val_labels, search_range=(0.1, 0.9, 0.02)):
        """寻找每个类别的最佳阈值"""
        best_thresholds = []
        per_class_metrics = []
        
        print("\n" + "-"*50)
        print("优化各类别阈值:")
        print("-"*50)
        
        for i in range(8):
            best_f1 = 0
            best_th = 0.5
            best_precision = 0
            best_recall = 0
            
            for th in np.arange(search_range[0], search_range[1], search_range[2]):
                preds = (val_probs[:, i] > th).astype(int)
                
                if preds.sum() > 0:
                    precision = precision_score(val_labels[:, i], preds, zero_division=0)
                    recall = recall_score(val_labels[:, i], preds, zero_division=0)
                    f1 = f1_score(val_labels[:, i], preds, zero_division=0)
                else:
                    precision, recall, f1 = 0, 0, 0
                
                if f1 > best_f1:
                    best_f1 = f1
                    best_th = th
                    best_precision = precision
                    best_recall = recall
            
            best_thresholds.append(best_th)
            per_class_metrics.append({
                'threshold': best_th,
                'f1': best_f1,
                'precision': best_precision,
                'recall': best_recall
            })
            
            print(f"类别 {i:2} {self.class_names[i]:15}: 阈值={best_th:.2f}, "
                  f"F1={best_f1:.3f}, P={best_precision:.3f}, R={best_recall:.3f}")
        
        self.best_thresholds = best_thresholds
        self.threshold_history.append(best_thresholds)
        
        return best_thresholds, per_class_metrics

print("✅ 阈值优化器定义完成")

In [ ]:
# ==================== 单元格5: 数据分析函数 ====================

def analyze_dataset_distribution():
    """分析数据集的类别分布"""
    print("\n" + "="*60)
    print("数据集分布分析")
    print("="*60)
    
    # 加载数据集（使用is_training=False避免数据增强）
    train_dataset = FundusDataset(train_dir, excel_dir, is_training=False)
    val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
    test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)
    
    def collect_labels(dataset, name):
        labels_list = []
        for i in tqdm(range(min(len(dataset), 5000)), desc=f"收集{name}标签"):
            _, labels, _ = dataset[i]
            labels_list.append(labels.numpy())
        labels_array = np.vstack(labels_list)
        
        pos_counts = labels_array.sum(axis=0)
        neg_counts = len(labels_array) - pos_counts
        
        print(f"\n{name}集分布 (共{len(labels_array)}个样本):")
        for i, (pos, neg) in enumerate(zip(pos_counts, neg_counts)):
            pos_pct = pos / len(labels_array) * 100
            print(f"  类别 {i:2} {class_names[i]:15}: 正样本={pos:4.0f} ({pos_pct:.1f}%), 负样本={neg:4.0f}")
        
        return labels_array
    
    print("\n正在收集标签...")
    train_labels = collect_labels(train_dataset, "训练")
    val_labels = collect_labels(val_dataset, "验证")
    test_labels = collect_labels(test_dataset, "测试")
    
    # 绘制分布图
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for ax, labels, name in zip(axes, [train_labels, val_labels, test_labels], ['训练集', '验证集', '测试集']):
        pos_counts = labels.sum(axis=0)
        bars = ax.bar(range(8), pos_counts, color='skyblue', edgecolor='navy', alpha=0.7)
        ax.set_title(f'{name} - 正样本分布', fontsize=14)
        ax.set_xlabel('类别', fontsize=12)
        ax.set_ylabel('样本数量', fontsize=12)
        ax.set_xticks(range(8))
        ax.set_xticklabels([c[:4] for c in class_names], rotation=45)
        
        # 在柱子上添加数值
        for bar, count in zip(bars, pos_counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    f'{int(count)}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('dataset_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # 确定少数类
    pos_counts = train_labels.sum(axis=0)
    minority_threshold = np.percentile(pos_counts, 40)  # 后40%为少数类
    minority_classes = [i for i, count in enumerate(pos_counts) if count <= minority_threshold]
    
    print(f"\n📊 分析结果:")
    print(f"  少数类阈值: {minority_threshold:.0f}个正样本")
    print(f"  少数类索引: {minority_classes}")
    print(f"  少数类名称: {[class_names[i] for i in minority_classes]}")
    
    return train_labels, val_labels, test_labels, minority_classes, train_dataset, val_dataset, test_dataset

print("✅ 数据分析函数定义完成")

In [ ]:
# ==================== 单元格6: 创建采样器 ====================

def create_adaptive_sampler(train_dataset, train_labels, minority_classes):
    """创建自适应采样器"""
    
    pos_counts = train_labels.sum(axis=0)
    neg_counts = len(train_labels) - pos_counts
    imbalance_ratio = neg_counts / (pos_counts + 1e-5)
    
    print("\n📊 类别不平衡比率:")
    for i, ratio in enumerate(imbalance_ratio):
        print(f"  类别 {i:2} {class_names[i]:15}: {ratio:.2f}")
    
    # 基础采样权重
    sample_weights = np.ones(len(train_dataset))
    
    # 为每个样本计算权重
    for i in tqdm(range(len(train_dataset)), desc="计算采样权重"):
        _, labels, _ = train_dataset[i]
        
        # 基于类别的逆频率
        weight = 1.0
        for class_idx in range(8):
            if labels[class_idx] == 1:
                class_weight = len(train_labels) / (pos_counts[class_idx] + 1)
                weight *= class_weight ** 0.3  # 降低指数避免权重过大
        
        # 少数类额外加成
        for class_idx in minority_classes:
            if labels[class_idx] == 1:
                weight *= 3.0
        
        sample_weights[i] = np.clip(weight, 0.1, 30)
    
    print(f"  采样权重范围: min={sample_weights.min():.2f}, max={sample_weights.max():.2f}")
    print(f"  权重均值: {sample_weights.mean():.2f}")
    print(f"  权重标准差: {sample_weights.std():.2f}")
    
    # 创建采样器
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights) * 2,  # 采样2倍数据
        replacement=True
    )
    
    return sampler, sample_weights

print("✅ 采样器函数定义完成")

In [ ]:
# ==================== 单元格7: 主训练函数 ====================

def train_model():
    print("\n" + "="*60)
    print("🚀 开始训练 ViT 模型")
    print("="*60)
    
    # 1. 分析数据分布
    train_labels, val_labels, test_labels, minority_classes, train_dataset, val_dataset, test_dataset = analyze_dataset_distribution()
    
    # 2. 设置少数类
    train_dataset.set_minority_classes(minority_classes)
    val_dataset.set_minority_classes(minority_classes)
    test_dataset.set_minority_classes(minority_classes)
    
    # 3. 创建采样器
    sampler, sample_weights = create_adaptive_sampler(train_dataset, train_labels, minority_classes)
    
    # 4. 创建DataLoader
    batch_size = 8  # 减小batch size
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=0,
        pin_memory=False,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    print(f"\n📦 DataLoader创建完成:")
    print(f"  训练批次: {len(train_loader)}")
    print(f"  验证批次: {len(val_loader)}")
    
    # 5. 加载模型
    print("\n🤖 加载预训练模型...")
    local_model_path = r'C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224'
    
    model = ViTForImageClassification.from_pretrained(
        local_model_path,
        num_labels=8,
        ignore_mismatched_sizes=True
    )
    model = model.to(device)
    print(f"✅ 模型加载成功！参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    
    # 6. 计算类别权重
    pos_counts = train_labels.sum(axis=0)
    neg_counts = len(train_labels) - pos_counts
    pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32).to(device)
    pos_weight = torch.clamp(pos_weight, 0.1, 10.0)
    
    print("\n⚖️ 类别权重:")
    for i, w in enumerate(pos_weight):
        print(f"  {class_names[i]}: {w:.3f}")
    
    # 7. 初始化损失函数和优化器
    criterion_bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    criterion_focal = FocalLoss(alpha=pos_weight, gamma=2.0)
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-5,  # 降低学习率
        weight_decay=0.01
    )
    
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)
    
    # 8. 初始化阈值优化器
    threshold_optimizer = ThresholdOptimizer(class_names)
    
    # 9. 训练循环
    epochs = 30
    best_val_f1 = 0
    best_model_state = None
    best_thresholds = None
    train_losses = []
    val_f1s = []
    
    print("\n" + "="*60)
    print("开始训练循环")
    print("="*60)
    
    for epoch in range(epochs):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch+1}/{epochs}")
        print('='*50)
        
        # 训练阶段
        model.train()
        train_loss_bce = 0
        train_loss_focal = 0
        train_steps = 0
        
        train_pbar = tqdm(train_loader, desc='Training')
        for images, labels, _ in train_pbar:
            try:
                images, labels = images.to(device), labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(images)
                
                loss_bce = criterion_bce(outputs.logits, labels)
                loss_focal = criterion_focal(outputs.logits, labels)
                loss = loss_bce + 0.5 * loss_focal  # 调整focal loss权重
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                train_loss_bce += loss_bce.item()
                train_loss_focal += loss_focal.item()
                train_steps += 1
                
                train_pbar.set_postfix({
                    'bce': f'{loss_bce.item():.4f}',
                    'focal': f'{loss_focal.item():.4f}',
                    'total': f'{loss.item():.4f}'
                })
                
            except Exception as e:
                print(f"训练批次错误: {e}")
                continue
        
        avg_loss = (train_loss_bce + train_loss_focal) / train_steps
        train_losses.append(avg_loss)
        
        # 验证阶段
        model.eval()
        all_probs = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels, _ in tqdm(val_loader, desc='Validating'):
                try:
                    images = images.to(device)
                    outputs = model(images)
                    probs = torch.sigmoid(outputs.logits).cpu().numpy()
                    all_probs.append(probs)
                    all_labels.append(labels.numpy())
                except Exception as e:
                    print(f"验证批次错误: {e}")
                    continue
        
        if len(all_probs) == 0:
            print("警告: 验证集没有有效数据")
            continue
        
        all_probs = np.vstack(all_probs)
        all_labels = np.vstack(all_labels)
        
        # 寻找最佳阈值
        best_thresholds, class_metrics = threshold_optimizer.find_best_thresholds(all_probs, all_labels)
        
        # 使用最佳阈值计算预测
        preds = np.zeros_like(all_probs)
        for i in range(8):
            preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)
        
        # 计算各类指标
        val_f1_macro = f1_score(all_labels, preds, average='macro', zero_division=0)
        val_f1_micro = f1_score(all_labels, preds, average='micro', zero_division=0)
        per_class_f1 = f1_score(all_labels, preds, average=None, zero_division=0)
        
        val_f1s.append(val_f1_macro)
        
        print(f"\n📊 验证结果:")
        print(f"  Macro F1: {val_f1_macro:.4f}")
        print(f"  Micro F1: {val_f1_micro:.4f}")
        print(f"  少数类F1: {np.mean([per_class_f1[i] for i in minority_classes if i < len(per_class_f1)]):.4f}")
        
        # 更新学习率
        scheduler.step()
        
        # 保存最佳模型
        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_thresholds = best_thresholds.copy()
            
            # 保存检查点
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': best_model_state,
                'val_f1': val_f1_macro,
                'per_class_f1': per_class_f1,
                'best_thresholds': best_thresholds,
                'class_metrics': class_metrics
            }
            torch.save(checkpoint, 'best_model_debug.pth')
            
            print(f"  ✅ 保存新最佳模型! F1={val_f1_macro:.4f}")
    
    # 绘制训练曲线
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(train_losses, 'b-', label='Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss Curve')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(val_f1s, 'r-', label='Validation Macro F1')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('F1 Score')
    ax2.set_title('Validation F1 Curve')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return model, best_model_state, best_thresholds, test_loader, minority_classes

print("✅ 主训练函数定义完成")

In [ ]:
# ==================== 单元格8: 测试函数 ====================

def test_model(model, best_model_state, best_thresholds, test_loader, minority_classes):
    """测试最终模型"""
    print("\n" + "="*60)
    print("🧪 最终测试")
    print("="*60)
    
    # 加载最佳模型
    model.load_state_dict(best_model_state)
    model = model.to(device)
    model.eval()
    
    # 测试集评估
    test_probs = []
    test_labels_list = []
    test_names = []
    
    with torch.no_grad():
        for images, labels, names in tqdm(test_loader, desc='Testing'):
            try:
                images = images.to(device)
                outputs = model(images)
                probs = torch.sigmoid(outputs.logits).cpu().numpy()
                test_probs.append(probs)
                test_labels_list.append(labels.numpy())
                test_names.extend(names)
            except Exception as e:
                print(f"测试批次错误: {e}")
                continue
    
    if len(test_probs) == 0:
        print("错误: 测试集没有有效数据")
        return
    
    test_probs = np.vstack(test_probs)
    test_labels = np.vstack(test_labels_list)
    
    # 使用最佳阈值
    test_preds = np.zeros_like(test_probs)
    for i in range(8):
        test_preds[:, i] = (test_probs[:, i] > best_thresholds[i]).astype(int)
    
    # 计算最终指标
    test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    test_f1_micro = f1_score(test_labels, test_preds, average='micro', zero_division=0)
    per_class_test_f1 = f1_score(test_labels, test_preds, average=None, zero_division=0)
    
    print(f"\n📊 最终测试结果:")
    print(f"  Macro F1: {test_f1_macro:.4f}")
    print(f"  Micro F1: {test_f1_micro:.4f}")
    print(f"  少数类平均F1: {np.mean([per_class_test_f1[i] for i in minority_classes if i < len(per_class_test_f1)]):.4f}")
    
    print("\n📈 各类别详细指标:")
    for i in range(8):
        precision = precision_score(test_labels[:, i], test_preds[:, i], zero_division=0)
        recall = recall_score(test_labels[:, i], test_preds[:, i], zero_division=0)
        f1 = per_class_test_f1[i]
        print(f"  {class_names[i]:15}: F1={f1:.3f}, P={precision:.3f}, R={recall:.3f}")
    
    # 绘制混淆矩阵
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for i in range(8):
        cm = confusion_matrix(test_labels[:, i], test_preds[:, i])
        sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues', 
                    xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
        axes[i].set_title(f'{class_names[i]}', fontsize=12)
        axes[i].set_xlabel('Predicted')
        axes[i].set_ylabel('Actual')
    
    plt.suptitle('Confusion Matrices for Each Class', fontsize=16)
    plt.tight_layout()
    plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # 保存详细结果
    results = {
        'test_f1_macro': test_f1_macro,
        'test_f1_micro': test_f1_micro,
        'per_class_f1': per_class_test_f1,
        'per_class_precision': [precision_score(test_labels[:, i], test_preds[:, i], zero_division=0) for i in range(8)],
        'per_class_recall': [recall_score(test_labels[:, i], test_preds[:, i], zero_division=0) for i in range(8)],
        'best_thresholds': best_thresholds,
        'minority_classes': minority_classes
    }
    
    # 保存为CSV
    results_df = pd.DataFrame({
        'Class': class_names,
        'F1_Score': per_class_test_f1,
        'Precision': results['per_class_precision'],
        'Recall': results['per_class_recall'],
        'Threshold': best_thresholds
    })
    results_df.to_csv('test_results.csv', index=False, encoding='utf-8-sig')
    print("\n💾 测试结果已保存到 test_results.csv")
    
    return results

print("✅ 测试函数定义完成")

In [ ]:
# ==================== 单元格9: 运行训练 ====================

# 运行训练
print("开始训练...")
model, best_model_state, best_thresholds, test_loader, minority_classes = train_model()

# 测试模型
if best_model_state is not None:
    results = test_model(model, best_model_state, best_thresholds, test_loader, minority_classes)
    print(f"\n🎉 训练完成！最佳Macro F1: {results['test_f1_macro']:.4f}")
else:
    print("❌ 训练失败，没有保存最佳模型")

In [ ]:
# ==================== 单元格10: 可选 - 加载已有模型进行测试 ====================

def load_and_test(checkpoint_path='best_model_debug.pth'):
    """加载已有模型进行测试"""
    print(f"\n📂 加载模型: {checkpoint_path}")
    
    # 加载检查点
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    print(f"  训练轮次: {checkpoint.get('epoch', 'N/A')}")
    print(f"  验证F1: {checkpoint.get('val_f1', 'N/A'):.4f}")
    
    # 重新加载数据
    print("\n📊 重新加载数据...")
    train_dataset = FundusDataset(train_dir, excel_dir, is_training=False)
    val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
    test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)
    
    # 获取少数类
    train_labels = []
    for i in tqdm(range(min(len(train_dataset), 1000)), desc="收集标签"):
        _, labels, _ = train_dataset[i]
        train_labels.append(labels.numpy())
    train_labels = np.vstack(train_labels)
    pos_counts = train_labels.sum(axis=0)
    minority_threshold = np.percentile(pos_counts, 40)
    minority_classes = [i for i, count in enumerate(pos_counts) if count <= minority_threshold]
    
    # 创建test loader
    test_loader = DataLoader(
        test_dataset,
        batch_size=8,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    # 加载模型
    local_model_path = r'C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224'
    model = ViTForImageClassification.from_pretrained(
        local_model_path,
        num_labels=8,
        ignore_mismatched_sizes=True
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # 测试
    results = test_model(
        model, 
        checkpoint['model_state_dict'], 
        checkpoint['best_thresholds'], 
        test_loader, 
        minority_classes
    )
    
    return results

# 如果需要加载已有模型测试，取消下面的注释
# results = load_and_test('best_model_debug.pth')